# 📘 Agentic AI Roadmap — Balaji Chippada
## Module 3.1 · UI vs API — The Hinge Moment

> **Roadmap**: [https://ch-balaji.github.io/ai-engineer-roadmap/](https://ch-balaji.github.io/ai-engineer-roadmap/)

---

### 🗺️ What this module is about

This is the **turning point** where you stop being an AI *user* and start becoming an AI *builder*.

You'll understand **why the same AI gives different results** in a chat UI vs API, what's happening behind the scenes that you never see, and why every real production application is built on the API — not the chat interface.

---

### 📑 Topics Covered

| # | Topic |
|---|-------|
| 01 | Same prompt, same model, different output — why? |
| 02 | System prompts you don't see |
| 03 | Skills/tools the chat UI calls silently |
| 04 | Why production work happens via API |

---
## 🔶 Topic 01 — Same Prompt, Same Model, Different Output — Why?

### 🤔 The Puzzle

You type this in **Claude.ai** (chat UI):
> *"Summarize this email and reply politely"*

You send the **exact same text** via the API.

The chat UI gives a nicely formatted, warm, structured reply.  
The API gives something shorter, raw, or differently structured.

**Same model. Same words. Different output.** Why? 🤯

---

### 💡 The Answer

The chat UI **secretly adds things before your message reaches the model**:

- 🕵️ A hidden **system prompt** (instructions on how to behave)
- 📅 Today's date injected automatically
- 🧠 Conversation memory (previous messages)
- 🎭 A persona ("You are a helpful assistant...")
- 📋 Output format rules ("Always use bullet points")

The **API gives you a blank slate.** No hidden setup. Just raw model power.

---

### 🍽️ Real-World Analogy

| | Chat UI | API |
|---|---|---|
| **Like a...** | Restaurant | Grocery Store |
| **What you get** | Chef already has recipe, tools ready, knows your taste | Raw ingredients — you cook everything |
| **Control** | You just say "pasta" | You decide every step |
| **Output** | Always polished & consistent | Depends entirely on what YOU set up |

> ✏️ **Key Takeaway**: The model (the AI brain) is the same. What's different is the **invisible kitchen setup** around it.

---
## 🔶 Topic 02 — System Prompts You Don't See

### 🧱 What is a System Prompt?

Every AI conversation has **two layers**:

```
┌──────────────────────────────────────────┐
│  SYSTEM PROMPT  (hidden — you never see) │  ← Business rules, persona, guardrails
├──────────────────────────────────────────┤
│  USER MESSAGE   (what you type)          │  ← Your actual question
└──────────────────────────────────────────┘
```

The AI **reads the system prompt first**, then your message. It shapes every single response.

---

### ✈️ Real-World Example

You're chatting with an airline support bot and ask:
> *"Can I get a refund?"*

**What you see:** A polite, policy-aware reply about refunds.

**What actually happened before your message:**

```
HIDDEN SYSTEM PROMPT (you never see this):
"You are a helpful support agent for FlyFast Airlines.
 Always be polite. Only discuss topics related to our airline.
 Never discuss competitor airlines.
 Refunds are allowed within 24 hours of booking only."
```

That hidden prompt is why the bot behaves in a specific, bounded way — the underlying model could answer anything, but the system prompt limits and shapes it.

---

### 🏢 In Production — YOU write the system prompt

When you use the **API**, you are now the one writing the system prompt. This is your **product's brain**. It defines:

- ✅ What the AI can and cannot talk about
- 🎭 The tone and personality
- 📜 Business rules and policies
- 🛡️ Safety guardrails

In [ ]:
# 📦 Install the Anthropic SDK first (run once)
# !pip install anthropic

import anthropic

client = anthropic.Anthropic(api_key="your-api-key")  # 🔑 Replace with your key

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,

    # ✍️ THIS is where YOU write the hidden system prompt
    # In the chat UI, Anthropic writes this — you never see it
    # In the API, YOU control this completely
    system="""
        You are a polite customer support agent for FlyFast Airlines.
        Only answer airline-related questions.
        Refunds are allowed within 24 hours of booking only.
        Always end your reply with: 'Safe travels! — FlyFast Team'
    """,

    messages=[
        {"role": "user", "content": "Can I get a refund for my ticket?"}
    ]
)

print(response.content[0].text)

### 🔍 Code Breakdown

| Part | What it does |
|---|---|
| `system="..."` | Your hidden instruction — like whispering to the AI before the user speaks |
| `messages=[{...}]` | The actual conversation (user's question) |
| `model=` | Which AI brain to use |
| `max_tokens=` | Max length of reply (also controls cost) |

> ✏️ **Key Takeaway**: Chat UI = Anthropic's system prompt (you can't change it). API = YOUR system prompt. This is where your product's identity lives.

---
## 🔶 Topic 03 — Skills/Tools the Chat UI Calls Silently

### 🛠️ What are "Tools" in AI?

Modern AI models aren't just text generators — they can **use tools** to take action or get information:

- 🔍 Search the web
- 🧮 Run a calculator
- 📄 Read a document
- 📧 Send an email
- 🗄️ Query a database

When you use **Claude.ai or ChatGPT**, the UI silently decides *when* to use these tools and *which ones*. You don't see it happening.

---

### 🌦️ Real-World Example

You type in Claude.ai:
> *"What's the weather in Tirupati today?"*

You get a weather answer. But Claude doesn't magically know this — it **silently called a web search tool**, got the result, and wove it into the reply. You never saw the search happen.

---

### ⚙️ The Agent Loop (How Tools Actually Work)

This is the **most important concept** in agentic AI:

```
┌─────────────────────────────────────────────────────┐
│                   THE AGENT LOOP                    │
│                                                     │
│  User Question                                      │
│       │                                             │
│       ▼                                             │
│    AI Model  ──── "I need to use tool X"            │
│       │                                             │
│       ▼                                             │
│  YOUR CODE runs the tool  ◄── AI does NOT run it!  │
│       │                                             │
│       ▼                                             │
│  Result sent back to AI                             │
│       │                                             │
│       ▼                                             │
│  AI gives final answer to user ✅                   │
└─────────────────────────────────────────────────────┘
```

> ⚠️ **Critical**: The AI *requests* the tool call. YOUR code *executes* it. The AI itself never directly runs code or queries databases.

---

### 🏥 Production Example — HR Chatbot Tools

An HR chatbot can be given these tools by the builder:

| Tool | What it does |
|---|---|
| `get_leave_balance` | Queries HR database for employee's leave days |
| `submit_leave_request` | Writes a new leave entry to the database |
| `send_email_confirmation` | Calls email API to notify the employee |

The AI orchestrates which tool to call and when. You can't do any of this through a chat UI — only via the API.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key="your-api-key")

# ──────────────────────────────────────────────
# STEP 1: Define your tools (the AI's toolbox)
# ──────────────────────────────────────────────
tools = [
    {
        "name": "calculate",
        "description": "Use this to do any math calculation",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression to evaluate, e.g. '25 * 4 + 10'"
                }
            },
            "required": ["expression"]
        }
    }
]

# ──────────────────────────────────────────────
# STEP 2: Send user message WITH the toolbox
# ──────────────────────────────────────────────
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=tools,           # 🧰 Give the AI a toolbox
    messages=[
        {"role": "user", "content": "What is 347 multiplied by 82?"}
    ]
)

# ──────────────────────────────────────────────
# STEP 3: AI responds asking to USE the tool
# ──────────────────────────────────────────────
# response.content will have a tool_use block:
# { "type": "tool_use", "name": "calculate", "input": {"expression": "347 * 82"} }
print(response.content)

# ──────────────────────────────────────────────
# STEP 4: YOUR code actually runs the tool
# ──────────────────────────────────────────────
def run_calculator(expression: str) -> str:
    """Your code that actually does the math"""
    result = eval(expression)  # In production, use a safe math parser
    return str(result)

# Simulate extracting tool call from response
tool_call = response.content[0]  # This would be the tool_use block
# result = run_calculator(tool_call.input["expression"])  # → "28454"

# ──────────────────────────────────────────────
# STEP 5: Send result back → AI gives final answer
# ──────────────────────────────────────────────
# (Send tool result back in messages array — AI then replies to the user)

### 🔍 Code Breakdown

| Step | What happens |
|---|---|
| `tools=[{...}]` | You define what tools the AI can *request* to use |
| AI response | Contains a `tool_use` block (AI saying "please run this for me") |
| `run_calculator()` | YOUR code that actually executes the tool |
| Send result back | AI uses the result to formulate the final user-facing reply |

> ✏️ **Key Takeaway**: In the chat UI, the platform decides which tools exist. With the API, YOU decide — including tools that connect to YOUR private databases, YOUR APIs, YOUR systems.

---
## 🔶 Topic 04 — Why Production Work Happens via API

### 🏗️ The Core Reason

| Need | Chat UI | API |
|---|---|---|
| Serve 10,000 users simultaneously | ❌ Impossible | ✅ Built for this |
| Connect to your private database | ❌ No access | ✅ Full control |
| Automate without human typing | ❌ Requires a human | ✅ Fully automated |
| Control cost per call | ❌ Not possible | ✅ Token-level control |
| Apply your business rules | ❌ Limited | ✅ Via system prompt |
| Integrate into your existing app | ❌ Not possible | ✅ Just an API call |

---

### 📧 Real Production Example — Auto Email Reply System

**Scenario**: You run an e-commerce store. 500 customer support emails arrive every day. You want AI to draft professional replies automatically — no human needed for first response.

**With Chat UI**: Impossible. Someone has to manually paste each email. 500 times. Every day.

**With API**: Your backend calls the function once per email. Done automatically. Scales to 50,000 emails if needed.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key="your-api-key")

def generate_email_reply(customer_email: str, customer_name: str, issue_type: str) -> str:
    """
    🏭 This is a PRODUCTION function.
    Called automatically by your backend when a support ticket arrives.
    No human is typing anything. Fully automated.
    Can handle thousands of emails per day.
    """

    # ✍️ YOUR business rules baked into the system prompt
    system_prompt = """
        You are a support agent for CloudStore, an e-commerce platform.
        Always address the customer by their first name.
        Be empathetic, professional, and concise.
        Refund policy: 7-day return window.
        Delivery time: 3-5 business days.
        Sign off every email as: 'Team CloudStore Support'
    """

    # 🤖 Programmatically constructed — no human typing this
    user_message = f"""
        Customer Name: {customer_name}
        Issue Type: {issue_type}
        Customer Email: {customer_email}

        Write a professional reply to this customer.
    """

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,           # 💰 Controls cost — shorter = cheaper
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_message}
        ]
    )

    return response.content[0].text


# ─────────────────────────────────────────────────────────
# This runs automatically when a ticket comes in from your
# database/CRM — no human involvement needed
# ─────────────────────────────────────────────────────────
reply = generate_email_reply(
    customer_email="I ordered 3 days ago and still haven't received my package. Very disappointed.",
    customer_name="Ravi Kumar",
    issue_type="Delivery Delay"
)

print(reply)
# → "Dear Ravi, Thank you for reaching out...[professional, policy-aware reply]..."
#   "Safe travels! — Team CloudStore Support"

### 🔍 Code Breakdown

| Part | Why it matters in production |
|---|---|
| `def generate_email_reply(...)` | A reusable function your server calls — no human needed |
| `system_prompt` | YOUR business rules — refund policy, tone, sign-off |
| `f"Customer: {customer_name}"` | Data pulled from your DB, fed programmatically |
| `max_tokens=500` | Hard cost cap — you know exactly what each reply costs |
| Function can be called in a loop | Process 10,000 emails — same code, zero extra effort |

---

### 💰 Cost Control — A Production Superpower

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key="your-api-key")

response = client.messages.create(
    model="claude-haiku-4-5-20251001",   # 💡 Cheaper model for simple tasks
    max_tokens=150,                        # 💡 Strict output limit = cost control
    system="Summarize customer complaints in 2 sentences max.",
    messages=[
        {"role": "user", "content": "Customer said their package was late and support was unhelpful."}
    ]
)

# ✅ After every API call, you see EXACTLY what you spent
print(f"Input tokens used  : {response.usage.input_tokens}")
print(f"Output tokens used : {response.usage.output_tokens}")
print(f"Reply              : {response.content[0].text}")

# In production: log this per user, per feature, per day
# → You know your AI costs down to the rupee

### 💡 Model Selection Strategy in Production

| Task Complexity | Model to Use | Why |
|---|---|---|
| Simple classification, short summaries | `claude-haiku` | Fastest & cheapest |
| Customer support, drafting replies | `claude-sonnet` | Balanced quality + cost |
| Complex reasoning, legal/medical analysis | `claude-opus` | Most capable, higher cost |

> ✏️ **Key Takeaway**: With the Chat UI, you have zero visibility into cost. With the API, you track every token, choose the right model per task, and optimize intelligently.

---
## 🧠 The Mental Model — Passenger vs Driver

```
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   CHAT UI  🚗  You are the PASSENGER                           │
│                                                                 │
│   • Comfortable, easy, fast to start                           │
│   • You go where the car (platform) takes you                  │
│   • No control over the route, speed, or fuel                  │
│   • Great for: personal use, exploration, learning              │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   API  🏎️  You are the DRIVER (+ mechanic + navigator)         │
│                                                                 │
│   • Full control: route, speed, fuel, passenger rules          │
│   • You set the system prompt (the GPS destination)            │
│   • You define the tools (the car's special features)          │
│   • You control cost (the fuel budget)                         │
│   • Great for: products, automation, real business impact       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```